#### Data Modelling by Group 13

## 1. Setup and Imports

In [16]:
import shutil
import os

print("Unzipping dataset safely...")
shutil.unpack_archive('dataset.zip', '/content/dataset')
print("Done! Dataset is ready.")

Unzipping dataset safely...
Done! Dataset is ready.


In [17]:
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.applications import ResNet50, DenseNet121, MobileNetV3Large
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import time
import os

## 2. Data Pipeline Configuration
Configuring the Keras dataset pipeline to load the animal images from the directory, automatically resize them to 224x224 (to match the required CNN input dimensions), and batch them for optimal training performance.

In [22]:
# Set path to the dataset
dataset_dir = './dataset'
train_dir = os.path.join(dataset_dir, 'train')
val_dir = os.path.join(dataset_dir, 'val')
test_dir = os.path.join(dataset_dir, 'test')

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# 1. Load Training Data
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical' # Required for multi-class categorization
)

# 2. Load Validation Data
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

# 3. Load Testing Data (for the Analyst to use later)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='categorical'
)

# Print out the classes that have been found
class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"\nAwesome! Found {num_classes} Animal Subspecies classes: {class_names}")

# Optimize data loading speed
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)


Found 3348 files belonging to 7 classes.
Found 717 files belonging to 7 classes.
Found 722 files belonging to 7 classes.

Awesome! Found 7 Animal Subspecies classes: ['bengal', 'british_shorthair', 'maine_coon', 'persian', 'ragdoll', 'siamese', 'sphynx']


## 3. Transfer Learning Model Builder
This function dynamically builds the three required CNN architectures: `ResNet50`, `DenseNet121`, and `MobileNetV3`. We freeze the base layers to utilize pre-trained ImageNet weights and attach a custom classification head for our 7 cat subspecies classes. The models are compiled to track both **Accuracy** and **Mean Average Precision (mAP)**.


In [ ]:
def build_model(model_name, num_classes):
    # Setup the input layer
    inputs = tf.keras.Input(shape=(224, 224, 3))

    # 1. Pick the base model requested by the rubric & apply its specific preprocessing
    if model_name == 'ResNet50':
        x = tf.keras.applications.resnet50.preprocess_input(inputs)
        base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=x)
    elif model_name == 'DenseNet121':
        x = tf.keras.applications.densenet.preprocess_input(inputs)
        base_model = DenseNet121(weights='imagenet', include_top=False, input_tensor=x)
    elif model_name == 'MobileNetV3':
        x = tf.keras.applications.mobilenet_v3.preprocess_input(inputs)
        base_model = MobileNetV3Large(weights='imagenet', include_top=False, input_tensor=x)
    else:
        raise ValueError("Invalid model name!")

    # 2. Freeze the base model (Transfer Learning)
    base_model.trainable = False

    # 3. Add our custom classification head for the Animal Subspecies
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu')(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    # 4. Put it all together
    model = Model(inputs=base_model.input, outputs=predictions)

    # Compile the model with metrics required by rubric (Accuracy and PR-AUC as mAP)
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(curve='PR', name='mAP')])

    return model


## 3.5 Speed Test

In [ ]:
print("--- Running 1-Epoch Speed Test on ResNet50 ---")

# 1. Build a temporary ResNet50 model
test_model = build_model('ResNet50', num_classes)

# 2. Start timer
start = time.time()

# 3. Train for exactly ONE epoch
test_model.fit(train_dataset, validation_data=validation_dataset, epochs=1)

# 4. End timer and calculate estimates
end = time.time()
one_epoch_time = end - start

print("\n" + "="*50)
print(f"One epoch took {one_epoch_time:.2f} seconds.")
print(f"Estimated time for 50 epochs (1 model): {(one_epoch_time * 50) / 60:.2f} minutes.")
print(f"Estimated time for all 3 models (150 epochs): {(one_epoch_time * 150) / 60 / 60:.2f} hours.")
print("="*50 + "\n")

--- Running 1-Epoch Speed Test on ResNet50 ---
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
105/105 ━━━━━━━━━━━━━━━━━━━━ 41s 249ms/step - accuracy: 0.8303 - loss: 0.5328 - mAP: 0.9085 - val_accuracy: 0.9010 - val_loss: 0.2939 - val_mAP: 0.9681

One epoch took 40.86 seconds.
Estimated time for 50 epochs (1 model): 34.05 minutes.
Estimated time for all 3 models (150 epochs): 1.70 hours.



## 4. Model Training & Evaluation

In [ ]:
# from tensorflow.keras.callbacks import ModelCheckpoint

# models_to_train = ['ResNet50', 'DenseNet121', 'MobileNetV3']
# epochs = 50
# training_results = {}

# for name in models_to_train:
#     print(f"\n{'='*50}\nStarting Training for {name}...\n{'='*50}")

#     # Build model dynamically
#     model = build_model(name, num_classes)

#     # This saves the model continuously during training.
#     # If the laptop dies, the best version still will be saved.
#     checkpoint = ModelCheckpoint(
#         filepath=f'{name}_best_model.keras', # Using modern .keras format
#         monitor='val_accuracy',              # Save when validation accuracy improves
#         save_best_only=True,
#         verbose=1
#     )

#     # Start timer
#     start_time = time.time()

#     # Run the Training (added callbacks=[checkpoint])
#     history = model.fit(
#         train_dataset,
#         validation_data=validation_dataset,
#         epochs=epochs,
#         callbacks=[checkpoint]
#     )

#     # End timer
#     end_time = time.time()
#     total_time = end_time - start_time

#     print(f"\n>>> {name} Total Training Time: {total_time/60:.2f} minutes <<<")

#     # Save results
#     training_results[name] = {
#         'history': history.history,
#         'time_seconds': total_time
#     }

#     # Save the final version just in case
#     model.save(f'{name}_final_model.keras')



Starting Training for ResNet50...
Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step - accuracy: 0.6810 - loss: 1.4136 - mAP: 0.7369
Epoch 1: val_accuracy improved from None to 0.91771, saving model to ResNet50_best_model.keras

Epoch 1: finished saving model to ResNet50_best_model.keras
105/105 ━━━━━━━━━━━━━━━━━━━━ 34s 215ms/step - accuracy: 0.8309 - loss: 0.6514 - mAP: 0.9023 - val_accuracy: 0.9177 - val_loss: 0.2580 - val_mAP: 0.9681
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9441 - loss: 0.1675 - mAP: 0.9834
Epoch 2: val_accuracy did not improve from 0.91771
105/105 ━━━━━━━━━━━━━━━━━━━━ 11s 101ms/step - accuracy: 0.9480 - loss: 0.1581 - mAP: 0.9864 - val_accuracy: 0.9052 - val_loss: 0.2987 - val_mAP: 0.9666
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9654 - loss: 0.1080 - mAP: 0.9929
Epoch 3: val_accuracy improved from 0.91771 to 0.93027, saving model to ResNet50_best_model.keras

Epoch 3: finished saving model to ResNet50_best_mod

## 4.1 Model Training (with CSV History)
Running the training loop a second time with `ModelCheckpoint` and `CSVLogger`. This ensures the `.keras` model weights and the 50-epoch training history (loss, accuracy, mAP) are safely saved to the hard drive for the Data Analyst to visualize.


In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger # <-- Added CSVLogger

models_to_train = ['ResNet50', 'DenseNet121', 'MobileNetV3']
epochs = 50
training_results = {}

for name in models_to_train:
    print(f"\n{'='*50}\nStarting Training for {name}...\n{'='*50}")

    model = build_model(name, num_classes)

    # 1. Saves the best model weights
    checkpoint = ModelCheckpoint(
        filepath=f'{name}_best_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

    # 2. Saves the history to a CSV file automatically
    csv_logger = CSVLogger(f'{name}_training_history.csv', append=False)

    start_time = time.time()

    # Run the Training (Notice both callbacks are added)
    history = model.fit(
        train_dataset,
        validation_data=validation_dataset,
        epochs=epochs,
        callbacks=[checkpoint, csv_logger]
    )

    end_time = time.time()
    total_time = end_time - start_time

    print(f"\n>>> {name} Total Training Time: {total_time/60:.2f} minutes <<<")

    training_results[name] = {
        'history': history.history,
        'time_seconds': total_time
    }



Starting Training for ResNet50...
Epoch 1/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.7156 - loss: 1.0714 - mAP: 0.7855
Epoch 1: val_accuracy improved from None to 0.92748, saving model to ResNet50_best_model.keras

Epoch 1: finished saving model to ResNet50_best_model.keras
105/105 ━━━━━━━━━━━━━━━━━━━━ 33s 213ms/step - accuracy: 0.8414 - loss: 0.5439 - mAP: 0.9158 - val_accuracy: 0.9275 - val_loss: 0.2390 - val_mAP: 0.9735
Epoch 2/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step - accuracy: 0.9419 - loss: 0.1719 - mAP: 0.9847
Epoch 2: val_accuracy did not improve from 0.92748
105/105 ━━━━━━━━━━━━━━━━━━━━ 26s 104ms/step - accuracy: 0.9444 - loss: 0.1739 - mAP: 0.9843 - val_accuracy: 0.9038 - val_loss: 0.3139 - val_mAP: 0.9618
Epoch 3/50
105/105 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9501 - loss: 0.1352 - mAP: 0.9903
Epoch 3: val_accuracy improved from 0.92748 to 0.93724, saving model to ResNet50_best_model.keras

Epoch 3: finished saving model to ResNet50_best_mod

## 5. Final Results Summary


In [ ]:
import pandas as pd
from IPython.display import display

# Create a clean table for the professor and Data Analyst
summary_data = []
for name, results in training_results.items():
    # Get the final epoch's metrics
    final_acc = results['history']['accuracy'][-1]
    final_val_acc = results['history']['val_accuracy'][-1]
    final_map = results['history']['mAP'][-1]
    time_mins = results['time_seconds'] / 60

    summary_data.append({
        'Model': name,
        'Training Time (mins)': f"{time_mins:.2f}",
        'Final Train Accuracy': f"{final_acc:.4f}",
        'Final Val Accuracy': f"{final_val_acc:.4f}",
        'Final mAP': f"{final_map:.4f}"
    })

# Display the table
df_summary = pd.DataFrame(summary_data)
print("\n" + "="*22)
print("FINAL MODEL COMPARISON")
print("="*22)
display(df_summary)


FINAL MODEL COMPARISON


,Model,Training Time (mins),Final Train Accuracy,Final Val Accuracy,Final mAP
0,ResNet50,9.81,1.0000,0.9386,1.0000
1,DenseNet121,9.49,0.9979,0.9303,0.9990
2,MobileNetV3,3.85,0.9997,0.9317,0.9995


## 6. Hyperparameter Tuning (Fine-Tuning)
We are performing Hyperparameter Tuning on our best-performing model, **DenseNet121**.

Our tuning process involves:
1. **Unfreezing** the base model so the weights can be adjusted.
2. **Tuning the Learning Rate** hyperparameter down to `1e-5` to safely adjust the pre-trained weights without destroying them.
3. **Fine-tuning** the model for an additional 5 epochs to squeeze out the maximum possible accuracy.

In [23]:
print("Starting Hyperparameter Tuning on our best model: DenseNet121...")

# 1. Load the best DenseNet model
tuned_model = tf.keras.models.load_model('DenseNet121_best_model.keras')

# 2. Unfreeze the base model so we can tune its internal weights
tuned_model.trainable = True

# 3. HYPERPARAMETER TUNING: We change the Learning Rate to a much smaller value
# (1e-5 instead of the default 1e-3) so we don't ruin the weights it already learned.
from tensorflow.keras.optimizers import Adam
tuned_optimizer = Adam(learning_rate=1e-5)

# Re-compile the model with the tuned hyperparameter
tuned_model.compile(optimizer=tuned_optimizer,
                    loss='categorical_crossentropy',
                    metrics=['accuracy', tf.keras.metrics.AUC(curve='PR', name='mAP')])

# 4. Train it for just 5 more epochs to squeeze out the maximum accuracy!
tuning_history = tuned_model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=5
)

# Save the final tuned model
tuned_model.save('DenseNet121_TUNED_model.keras')
print("Hyperparameter tuning complete!")


Starting Hyperparameter Tuning on our best model: DenseNet121...
Epoch 1/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 83s 497ms/step - accuracy: 0.9991 - loss: 0.0127 - mAP: 1.0000 - val_accuracy: 0.9470 - val_loss: 0.2098 - val_mAP: 0.9826
Epoch 2/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 94ms/step - accuracy: 0.9991 - loss: 0.0117 - mAP: 1.0000 - val_accuracy: 0.9456 - val_loss: 0.2103 - val_mAP: 0.9826
Epoch 3/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 95ms/step - accuracy: 0.9994 - loss: 0.0111 - mAP: 1.0000 - val_accuracy: 0.9456 - val_loss: 0.2112 - val_mAP: 0.9827
Epoch 4/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 0.9994 - loss: 0.0107 - mAP: 1.0000 - val_accuracy: 0.9442 - val_loss: 0.2116 - val_mAP: 0.9827
Epoch 5/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - accuracy: 0.9994 - loss: 0.0101 - mAP: 1.0000 - val_accuracy: 0.9456 - val_loss: 0.2124 - val_mAP: 0.9827
Hyperparameter tuning complete!
